In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config

Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load post-release Bronze table
df_post = spark.table(f"{BRONZE_PATH}.google_trends_postrelease_raw")

# Convert date column
df_post = df_post.withColumn("date", F.to_date(F.col("date")))

# Per film window
film_window = Window.partitionBy("film")

# Key metrics
df_post_silver = (df_post
    .withColumn("post_peak_score",
        F.max("interest_score").over(film_window))
    .withColumn("post_avg_score",
        F.avg("interest_score").over(film_window))
    .withColumn("days_after_release",
        F.datediff(F.col("date"), F.col("release_date").cast("date")))
)

# Collapse to one row per film
df_post_agg = (df_post_silver
    .groupBy("film", "release_date", "genre")
    .agg(
        F.max("interest_score").alias("post_peak_score"),
        F.avg("interest_score").alias("post_avg_score"),
        F.sum(F.when(F.col("interest_score") > 50, 1)
               .otherwise(0)).alias("days_above_50_post"),
        # First 30 days average — measures immediate drop-off
        F.avg(F.when(F.col("days_after_release") <= 30,
               F.col("interest_score"))).alias("avg_first30_post")
    )
)

print(f"✅ Post-release silver ready — {df_post_agg.count()} films")
display(df_post_agg)

✅ Post-release silver ready — 50 films


film,release_date,genre,post_peak_score,post_avg_score,days_above_50_post,avg_first30_post
Bridesmaids,2011-05-13,Comedy,100,25.24175824175824,8,40.645161290322584
Paranormal Activity,2009-10-16,Horror,100,17.439560439560438,10,39.38709677419355
Interstellar,2014-11-07,Sci-Fi,100,16.483516483516482,6,32.61290322580645
Gravity,2013-10-04,Sci-Fi,100,24.725274725274726,6,40.516129032258064
The Hangover,2009-06-05,Comedy,100,25.36263736263736,13,47.935483870967744
Gone Girl,2014-10-03,Thriller,100,18.175824175824175,6,33.774193548387096
The Conjuring,2013-07-19,Horror,100,13.197802197802197,4,29.580645161290324
Mad Max: Fury Road,2015-05-15,Action,100,17.21978021978022,7,33.61290322580645
The Social Network,2010-10-01,Drama,100,16.428571428571427,5,31.612903225806452
The Wolf of Wall Street,2013-12-25,Drama,100,27.978021978021978,14,48.96774193548387


In [0]:
(df_post_agg
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER_PATH}.trends_postrelease_silver")
)

print(f"✅ Saved to {SILVER_PATH}.trends_postrelease_silver")
print(f"   Rows written: {df_post_agg.count()}")

✅ Saved to bootcamp_students.tiffani_ceresrain_silver.trends_postrelease_silver
   Rows written: 50
